<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; font-size: 1.05em; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A; padding-bottom: .3em; }
.reveal h3 { color: #1A7A9A; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size: .75em; box-shadow: none; border-left: 3px solid #1A7A9A; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.alerta { background:#FDE8E8; border-left:4px solid #C0392B; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
</style>

## Análisis de Salida: Estado Estacionario
### T3.4 · Modelado de Sistemas bajo Incertidumbre
Universidad de los Andes · Ingeniería Industrial

## Agenda
1. Problema del período transitorio (warm-up)
2. Método de Welch — determinación gráfica de t_w
3. **Ejemplo 1:** M/M/1 estacionario con Batch Means
4. Common Random Numbers (CRN) — reducción de varianza
5. **Ejemplo 2:** M/M/1 vs M/D/1 con CRN
6. **Ejemplo 3:** Call center — comparación de 6 vs 7 agentes
7. Resumen del Módulo 3

## El problema del warm-up
<div class='defn'>
Toda simulación arranca desde un estado inicial X(0) que no es representativo del estado estacionario. Las observaciones del <strong>período transitorio</strong> sesgan la estimación.
</div>

$$E\left[\bar{Y}_m\right] = \theta + \underbrace{\frac{1}{m}\sum_{i=1}^{m}(E[Y_i] - \theta)}_{\text{sesgo de inicialización}}$$

<div class='alerta'>
El sesgo <strong>no desaparece</strong> al aumentar réplicas. Solo se elimina descartando el transitorio.
</div>

**Método de Welch:** promediar entre réplicas → suavizar con media móvil → identificar estabilización visual.

## Batch Means
<div class='defn'>
Una corrida larga (post warm-up) se divide en m lotes de b observaciones. Las medias de lote Ȳ₁,...,Ȳₘ se tratan como i.i.d. si b es suficientemente grande.

$$IC = \bar{Y}_m \pm t_{\alpha/2, m-1} \cdot \frac{S_B}{\sqrt{m}}$$
</div>

**Verificación de independencia:** Razón de Von Neumann
$$VN = \frac{\sum(\bar{Y}_{j+1} - \bar{Y}_j)^2}{(m-1) \cdot S_B^2} \approx 2 \text{ bajo independencia}$$

<div class='nota'>
Si VN << 2, los lotes están correlacionados → aumentar b (lotes más largos).
</div>

## Ejemplo 1 — M/M/1 estacionario con Batch Means
<div class='defn'>
Línea de producción 24/7: λ=4 pzas/min, μ=5 pzas/min, ρ=0.8.
Una corrida larga de 50,000 clientes. Descartar warm-up, dividir en lotes.
</div>

**Referencia analítica:** Wq = ρ/[μ(1−ρ)] = 0.80 min

**Plan:**
1. Método de Welch → determinar t_w
2. Batch Means → IC para Wq
3. Verificar independencia de lotes (Von Neumann)

In [ ]:
import simpy
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats as st

def mm1_largo(lam, mu, n_clientes, seed):
    """Simula M/M/1 y retorna vector de tiempos de espera por cliente."""
    np.random.seed(seed)
    esperas = []
    
    def cliente(env, srv):
        t_arr = env.now
        with srv.request() as req:
            yield req
            esperas.append(env.now - t_arr)
            yield env.timeout(np.random.exponential(1/mu))
    
    def llegadas(env, srv):
        i = 0
        while i < n_clientes:
            yield env.timeout(np.random.exponential(1/lam))
            env.process(cliente(env, srv))
            i += 1
    
    env = simpy.Environment()
    srv = simpy.Resource(env, capacity=1)
    env.process(llegadas(env, srv))
    env.run()
    return np.array(esperas)

lam, mu = 4, 5
Wq_ref = (lam/mu) / (mu * (1-lam/mu))  # 0.80

# ─── Método de Welch: k=10 réplicas de 10,000 clientes ───
k_welch = 10
n_welch = 10000
replicas_welch = np.array([mm1_largo(lam, mu, n_welch, seed=100+j) for j in range(k_welch)])

# Promedio entre réplicas para cada cliente i
min_len = min(len(r) for r in replicas_welch)
Y_bar_i = np.mean([r[:min_len] for r in replicas_welch], axis=0)

# Media móvil con diferentes ventanas
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(Y_bar_i[:3000], color='#BBDEFB', lw=0.5, alpha=0.6, label='Ȳᵢ (promedio entre réplicas)')
for w, col, ls in [(100, '#C8961E', '-'), (250, '#1A7A9A', '-'), (500, '#C62828', '--')]:
    smoothed = np.convolve(Y_bar_i, np.ones(2*w+1)/(2*w+1), mode='valid')
    axes[0].plot(range(w, w+len(smoothed))[:3000-w], smoothed[:3000-w], color=col, lw=2, ls=ls, label=f'w={w}')
axes[0].axhline(Wq_ref, ls=':', color='black', lw=1.5, label=f'θ = {Wq_ref:.2f}')
axes[0].axvline(3000, ls='--', color='gray', alpha=0.5)
axes[0].set_xlabel('Cliente i'); axes[0].set_ylabel('Wq suavizado')
axes[0].set_title('Método de Welch: detección del warm-up')
axes[0].legend(fontsize=7); axes[0].set_xlim(0, 5000)

# Media acumulada mostrando sesgo
media_acum = np.cumsum(Y_bar_i) / np.arange(1, len(Y_bar_i)+1)
axes[1].plot(media_acum, color='#1A7A9A', lw=1.5)
axes[1].axhline(Wq_ref, ls='--', color='#C62828', lw=2, label=f'θ = {Wq_ref:.2f}')
axes[1].axvline(5000, ls='--', color='gray', label='tw = 5,000')
axes[1].set_xlabel('Observaciones incluidas'); axes[1].set_ylabel('Media acumulada')
axes[1].set_title('Sesgo de inicialización'); axes[1].legend()
axes[1].set_xlim(0, min_len)

plt.tight_layout(); plt.show()
print(f'Warm-up elegido: tw = 5,000 clientes (conservador)')

In [ ]:
# ─── Batch Means: 1 corrida larga ───
W_largo = mm1_largo(lam, mu, 50000, seed=42)

tw = 5000  # warm-up
W_post = W_largo[tw:]  # observaciones post warm-up
n_obs = len(W_post)
m_lotes = 30
b_tam = n_obs // m_lotes

# Medias de lote
Y_batch = np.array([W_post[j*b_tam:(j+1)*b_tam].mean() for j in range(m_lotes)])

Ybar_B = Y_batch.mean()
SB = Y_batch.std(ddof=1)
t_c = st.t.ppf(0.975, m_lotes-1)
h_B = t_c * SB / np.sqrt(m_lotes)

# Razón de Von Neumann
VN = np.sum(np.diff(Y_batch)**2) / ((m_lotes-1) * SB**2)

print(f'═══ BATCH MEANS ═══')
print(f'Observaciones totales: {len(W_largo)}')
print(f'Warm-up descartado: {tw}')
print(f'Observaciones usadas: {n_obs}')
print(f'Lotes: m = {m_lotes}, tamaño b = {b_tam}')
print(f'\nȲ = {Ybar_B:.4f} min')
print(f'SB = {SB:.4f} min')
print(f'IC 95%: [{Ybar_B-h_B:.4f}, {Ybar_B+h_B:.4f}] min')
print(f'\nRef. analítica: Wq = {Wq_ref:.4f}')
print(f'¿IC contiene {Wq_ref}? {"Sí ✓" if Ybar_B-h_B <= Wq_ref <= Ybar_B+h_B else "No"}')
print(f'Error relativo: {abs(Ybar_B-Wq_ref)/Wq_ref*100:.1f}%')
print(f'\nRazón de Von Neumann: VN = {VN:.2f} (≈2 bajo independencia) {"✓" if 1.5 < VN < 2.5 else "⚠️"}')

# Gráfico de medias de lote
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(range(1, m_lotes+1), Y_batch, color='#1A7A9A', alpha=0.8)
axes[0].axhline(Ybar_B, color='#C62828', lw=2, ls='--', label=f'Ȳ = {Ybar_B:.3f}')
axes[0].axhline(Wq_ref, color='#2E7D32', lw=2, ls=':', label=f'θ = {Wq_ref:.3f}')
axes[0].axhspan(Ybar_B-h_B, Ybar_B+h_B, alpha=0.1, color='#C62828', label='IC 95%')
axes[0].set_xlabel('Lote j'); axes[0].set_ylabel('Ȳⱼ (min)')
axes[0].set_title(f'Medias de {m_lotes} lotes (b={b_tam})'); axes[0].legend(fontsize=8)

# Autocorrelación de medias de lote
from numpy import correlate as nc
y_c = Y_batch - Y_batch.mean()
acf_batch = np.correlate(y_c, y_c, 'full')
acf_batch = acf_batch[len(acf_batch)//2:] / acf_batch[len(acf_batch)//2]
axes[1].bar(range(len(acf_batch)), acf_batch, color='#1A7A9A', alpha=0.8)
axes[1].axhline(2/np.sqrt(m_lotes), ls='--', color='#C62828', label='±2/√m')
axes[1].axhline(-2/np.sqrt(m_lotes), ls='--', color='#C62828')
axes[1].set_xlabel('Lag'); axes[1].set_ylabel('ACF')
axes[1].set_title('Autocorrelación de medias de lote'); axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

## Common Random Numbers (CRN)
<div class='defn'>
Para comparar dos sistemas, usar la <strong>misma</strong> secuencia aleatoria en ambos. Esto introduce correlación positiva y reduce Var[Δ].

$$\text{Var}[\Delta_j] = \text{Var}[Y_j^{(1)}] + \text{Var}[Y_j^{(2)}] - 2\,\text{Cov}(Y_j^{(1)}, Y_j^{(2)})$$
</div>

Con CRN: Cov > 0 → Var[Δ] se reduce → IC más estrecho → menos réplicas necesarias.

<div class='alerta'>
⚠️ CRN requiere <strong>sincronización correcta</strong>: el mismo número aleatorio debe usarse para el mismo tipo de evento en ambos sistemas.
</div>

## Ejemplo 2 — M/M/1 vs M/D/1 con CRN
<div class='defn'>
Misma carga (λ=4, μ=5, ρ=0.8), diferente variabilidad de servicio:

- **Sistema A:** servicio exponencial (CV=1) → Wq = 0.80 min
- **Sistema B:** servicio determinista (CV=0) → Wq = 0.40 min

El servicio determinista reduce Wq exactamente a la mitad.
</div>

In [ ]:
def mm1_vs_md1_crn(lam, mu, n_clientes, seed, warmup=1000):
    """Simula M/M/1 y M/D/1 con CRN (mismas llegadas)."""
    np.random.seed(seed)
    
    # Generar llegadas y servicios por adelantado
    inter_arrivals = np.random.exponential(1/lam, n_clientes)
    services_exp = np.random.exponential(1/mu, n_clientes)
    service_det = 1/mu  # determinista
    
    # Simular M/M/1
    def sim_queue(services):
        esperas = []
        t_arr = 0
        t_salida = 0
        for i in range(n_clientes):
            t_arr += inter_arrivals[i]
            espera = max(0, t_salida - t_arr)
            esperas.append(espera)
            s = services[i] if isinstance(services, np.ndarray) else services
            t_salida = t_arr + espera + s
        return np.mean(esperas[warmup:])
    
    Wq_A = sim_queue(services_exp)
    Wq_B = sim_queue(np.full(n_clientes, service_det))
    return Wq_A, Wq_B

def mm1_vs_md1_indep(lam, mu, n_clientes, seed_A, seed_B, warmup=1000):
    """Simula M/M/1 y M/D/1 con semillas independientes."""
    np.random.seed(seed_A)
    ia_A = np.random.exponential(1/lam, n_clientes)
    sv_A = np.random.exponential(1/mu, n_clientes)
    
    np.random.seed(seed_B)
    ia_B = np.random.exponential(1/lam, n_clientes)
    
    def sim_q(ia, sv):
        esperas = []
        t_arr, t_sal = 0, 0
        for i in range(n_clientes):
            t_arr += ia[i]
            w = max(0, t_sal - t_arr)
            esperas.append(w)
            t_sal = t_arr + w + (sv[i] if isinstance(sv, np.ndarray) else sv)
        return np.mean(esperas[warmup:])
    
    return sim_q(ia_A, sv_A), sim_q(ia_B, np.full(n_clientes, 1/mu))

n_rep = 30
n_cli = 6000

# ─── Con CRN ───
results_crn = np.array([mm1_vs_md1_crn(lam, mu, n_cli, seed=2000+j) for j in range(n_rep)])
Wq_A_crn, Wq_B_crn = results_crn[:, 0], results_crn[:, 1]
Delta_crn = Wq_A_crn - Wq_B_crn

# ─── Sin CRN (independientes) ───
results_indep = np.array([mm1_vs_md1_indep(lam, mu, n_cli, 3000+j, 4000+j) for j in range(n_rep)])
Delta_indep = results_indep[:, 0] - results_indep[:, 1]

# Estadísticos
t_c30 = st.t.ppf(0.975, n_rep-1)

for name, delta in [('CRN', Delta_crn), ('Independientes', Delta_indep)]:
    m_d, s_d = delta.mean(), delta.std(ddof=1)
    h_d = t_c30 * s_d / np.sqrt(n_rep)
    print(f'\n═══ {name.upper()} ═══')
    print(f'Δ̄ = {m_d:.4f} min,  SΔ = {s_d:.4f} min')
    print(f'IC 95%: [{m_d-h_d:.4f}, {m_d+h_d:.4f}]')
    print(f'Semiancho: {h_d:.4f} min')
    print(f'¿IC excluye 0? {"Sí → diferencia significativa" if (m_d-h_d > 0 or m_d+h_d < 0) else "No"}')

print(f'\n═══ EFICIENCIA DE CRN ═══')
reduccion = 1 - Delta_crn.std()/Delta_indep.std()
print(f'Reducción de SΔ: {reduccion*100:.0f}%')
n_equiv = int(np.ceil(n_rep * (Delta_indep.std()/Delta_crn.std())**2))
print(f'Para igual IC sin CRN se necesitarían: {n_equiv} réplicas (vs {n_rep} con CRN)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Diferencias pareadas CRN vs independientes
axes[0].hist(Delta_crn, bins=10, color='#1A7A9A', alpha=0.7, label=f'CRN (SΔ={Delta_crn.std():.3f})')
axes[0].hist(Delta_indep, bins=10, color='#C8961E', alpha=0.5, label=f'Indep (SΔ={Delta_indep.std():.3f})')
axes[0].axvline(0.40, color='#C62828', lw=2, ls='--', label='Δ teórico = 0.40')
axes[0].set_xlabel('Δ = Wq(A) − Wq(B)'); axes[0].set_title('Distribución de Δ')
axes[0].legend(fontsize=8)

# Scatter CRN: correlación entre sistemas
axes[1].scatter(Wq_A_crn, Wq_B_crn, color='#1A7A9A', s=30, alpha=0.7)
corr = np.corrcoef(Wq_A_crn, Wq_B_crn)[0, 1]
axes[1].set_xlabel('Wq Sistema A (M/M/1)'); axes[1].set_ylabel('Wq Sistema B (M/D/1)')
axes[1].set_title(f'Correlación CRN: r = {corr:.3f}')

# IC comparativo
methods = ['CRN', 'Independientes']
deltas = [Delta_crn, Delta_indep]
colors_ic = ['#1A7A9A', '#C8961E']
for i, (name, delta, col) in enumerate(zip(methods, deltas, colors_ic)):
    m_d = delta.mean()
    h_d = t_c30 * delta.std(ddof=1) / np.sqrt(n_rep)
    axes[2].errorbar(i, m_d, yerr=h_d, fmt='o', color=col, capsize=10, capthick=2, markersize=8, label=name)
axes[2].axhline(0.40, color='#C62828', ls='--', label='Δ teórico')
axes[2].set_xticks([0, 1]); axes[2].set_xticklabels(methods)
axes[2].set_ylabel('Δ (min)'); axes[2].set_title('IC 95% de Δ: CRN vs Independientes')
axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

## Ejemplo 3 — Call center: 6 vs 7 agentes
<div class='defn'>
Call center 24/7. λ=45 llamadas/h, servicio Log-normal (E[S]=6.55 min, CV=0.63).
Comparar 6 vs 7 agentes con CRN. No existe fórmula exacta (M/G/c con Log-normal).
</div>

| | Alt. 1 (c=6) | Alt. 2 (c=7) |
|---|---|---|
| ρ teórico | 0.82 | 0.70 |
| Wq esperado | ~2 min | ~0.4 min |
| P(Wq>2 min) | ~28% | ~4% |

In [ ]:
def call_center(lam_h, mu_L, sigma_L, c, T_h, seed, warmup_h=500):
    """Simula call center M/G/c con servicio Log-normal."""
    np.random.seed(seed)
    esperas = []
    T_min = T_h * 60
    warmup_min = warmup_h * 60
    lam_min = lam_h / 60
    
    def cliente(env, srv):
        t_arr = env.now
        with srv.request() as req:
            yield req
            w = env.now - t_arr
            if env.now > warmup_min:  # solo post-warmup
                esperas.append(w)
            yield env.timeout(np.random.lognormal(mu_L, sigma_L))
    
    def llegadas(env, srv):
        while True:
            yield env.timeout(np.random.exponential(1/lam_min))
            env.process(cliente(env, srv))
    
    env = simpy.Environment()
    srv = simpy.Resource(env, capacity=c)
    env.process(llegadas(env, srv))
    env.run(until=T_min)
    
    if len(esperas) == 0:
        return 0, 0, 0
    
    Wq = np.mean(esperas)
    prob_gt2 = np.mean([w > 2 for w in esperas])
    return Wq, prob_gt2, len(esperas)

mu_L, sigma_L = 1.7, 0.6
lam_call = 45  # llamadas/h
T_sim = 1500   # horas (tras warm-up)
n_rep_cc = 40

# ─── CRN: misma semilla para ambos ───
print('═══ CALL CENTER: 6 vs 7 AGENTES (CRN, n=40) ═══\n')
res_6, res_7 = [], []
for j in range(n_rep_cc):
    seed_j = 5000 + j
    r6 = call_center(lam_call, mu_L, sigma_L, c=6, T_h=T_sim, seed=seed_j)
    r7 = call_center(lam_call, mu_L, sigma_L, c=7, T_h=T_sim, seed=seed_j)
    res_6.append(r6[:2])
    res_7.append(r7[:2])

res_6 = np.array(res_6)
res_7 = np.array(res_7)

# Resultados por alternativa
for name, res, c_val in [('Alt.1 (c=6)', res_6, 6), ('Alt.2 (c=7)', res_7, 7)]:
    for i, metric in enumerate(['Wq (min)', 'P(Wq>2)']):
        m = res[:, i].mean()
        s = res[:, i].std(ddof=1)
        h = st.t.ppf(0.975, n_rep_cc-1) * s / np.sqrt(n_rep_cc)
        print(f'{name} {metric:<12}: {m:.3f}  IC [{m-h:.3f}, {m+h:.3f}]')
    print()

# Diferencia pareada Wq
Delta_Wq = res_6[:, 0] - res_7[:, 0]
m_d = Delta_Wq.mean()
s_d = Delta_Wq.std(ddof=1)
h_d = st.t.ppf(0.975, n_rep_cc-1) * s_d / np.sqrt(n_rep_cc)
print(f'Δ(Wq) = {m_d:.3f} min,  IC [{m_d-h_d:.3f}, {m_d+h_d:.3f}]')
print(f'¿Significativa? {"Sí" if m_d-h_d > 0 else "No"} (IC excluye 0)')
print(f'\nAgregar 1 agente reduce Wq en ~{m_d:.1f} min y P(Wq>2) del {res_6[:,1].mean()*100:.0f}% al {res_7[:,1].mean()*100:.0f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Wq por alternativa
for i, (data, name, col) in enumerate([(res_6[:,0], 'c=6', '#C62828'), (res_7[:,0], 'c=7', '#1A7A9A')]):
    axes[0].hist(data, bins=10, color=col, alpha=0.6, label=f'{name}: Ȳ={data.mean():.2f}')
axes[0].set_xlabel('Wq (min)'); axes[0].set_title('Distribución de Wq por alternativa')
axes[0].legend()

# Diferencia pareada
axes[1].hist(Delta_Wq, bins=10, color='#1A7A9A', edgecolor='white', alpha=0.8)
axes[1].axvline(m_d, color='#C62828', lw=2, label=f'Δ̄ = {m_d:.2f}')
axes[1].axvline(0, color='black', lw=1.5, ls='--', label='Δ = 0')
axes[1].axvspan(m_d-h_d, m_d+h_d, alpha=0.15, color='#C62828', label='IC 95%')
axes[1].set_xlabel('Δ Wq (min)'); axes[1].set_title('Diferencia pareada (CRN)')
axes[1].legend(fontsize=8)

# Resumen visual
metrics_names = ['Wq (min)', 'P(Wq>2 min)']
for i, (name, idx) in enumerate(zip(metrics_names, [0, 1])):
    m6, m7 = res_6[:, idx].mean(), res_7[:, idx].mean()
    s6 = st.t.ppf(0.975, n_rep_cc-1) * res_6[:,idx].std(ddof=1) / np.sqrt(n_rep_cc)
    s7 = st.t.ppf(0.975, n_rep_cc-1) * res_7[:,idx].std(ddof=1) / np.sqrt(n_rep_cc)
    axes[2].errorbar(i-0.1, m6, yerr=s6, fmt='s', color='#C62828', capsize=8, markersize=8, label='c=6' if i==0 else '')
    axes[2].errorbar(i+0.1, m7, yerr=s7, fmt='o', color='#1A7A9A', capsize=8, markersize=8, label='c=7' if i==0 else '')
axes[2].set_xticks([0, 1]); axes[2].set_xticklabels(metrics_names)
axes[2].set_title('Comparación: 6 vs 7 agentes'); axes[2].legend()

plt.tight_layout(); plt.show()

print('\n📊 Recomendación: con 7 agentes, P(Wq>2 min) baja del ~28% al ~4%,')
print('   cumpliendo el estándar típico de nivel de servicio (≤5%).')

## Resumen del Módulo 3

| Tema | Pregunta central | Herramienta clave |
|---|---|---|
| **T3.1 Entradas** | ¿Qué distribución usar? | MLE + pruebas de ajuste + AIC |
| **T3.2 V&V** | ¿El modelo es correcto? | Trazas + IC vs. fórmulas |
| **T3.3 Terminal** | ¿Cuánto espera un cliente hoy? | Réplicas independientes |
| **T3.4 Estacionario** | ¿Cuánto espera a largo plazo? | Batch Means + CRN |

<div class='nota'>
Estos 4 pasos <strong>no son opcionales</strong>. Omitir alguno compromete la credibilidad de las conclusiones.
</div>

**Flujo completo:** Datos → Distribución (T3.1) → Modelo → V&V (T3.2) → Simulación → Análisis de salida (T3.3/T3.4) → Decisión